# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [ ]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [ ]:
# Load environment variables from the .env file at the repo root.
# override=True so edits to .env win over anything already set in the shell.
load_dotenv(dotenv_path="../../.env", override=True)

In [ ]:
# Preflight: prove every external service works before building anything on top of them.
# Reports all failures instead of stopping at the first one.

from openai import OpenAI
from tavily import TavilyClient

results = []

def check(label, fn):
    """Run one check, record pass/fail, never raise."""
    try:
        results.append((label, "PASS", fn()))
    except Exception as e:
        results.append((label, "FAIL", f"{type(e).__name__}: {e}"))

# 1. Which keys did .env actually give us? Print presence only, never values.
for key in ["OPENAI_API_KEY", "OPENAI_BASE_URL", "CHROMA_OPENAI_API_KEY", "TAVILY_API_KEY"]:
    val = os.getenv(key)
    results.append((f"env: {key}", "PASS" if val else "MISSING", f"{len(val)} chars" if val else "-"))

# 2. Chat endpoint. Uses OPENAI_BASE_URL automatically if it is set.
def chat_check():
    client = OpenAI()
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Reply with the single word: ok"}],
        max_tokens=5,
    )
    return r.choices[0].message.content.strip()

# 3. Embeddings via the OpenAI SDK. Same client, same key as chat.
def embed_sdk_check():
    client = OpenAI()
    r = client.embeddings.create(model="text-embedding-3-small", input="Gran Turismo")
    return f"{len(r.data[0].embedding)} dims"

# 4. Embeddings via Chroma's own function -- this is the one that matters.
#    Chroma builds a SEPARATE client, so a pass here proves nothing about check 3 and vice versa.
def embed_chroma_check():
    fn = embedding_functions.OpenAIEmbeddingFunction(
        api_key=os.getenv("OPENAI_API_KEY"),
        model_name="text-embedding-3-small",
    )
    return f"{len(fn(['Gran Turismo'])[0])} dims"

# 5. Web search.
def tavily_check():
    r = TavilyClient(api_key=os.getenv("TAVILY_API_KEY")).search("Mortal Kombat X PlayStation 5", max_results=1)
    return f"{len(r.get('results', []))} result(s)"

check("openai: chat", chat_check)
check("openai: embeddings (SDK)", embed_sdk_check)
check("chroma: embedding function", embed_chroma_check)
check("tavily: search", tavily_check)

results.append(("chromadb version", "INFO", chromadb.__version__))

width = max(len(r[0]) for r in results)
for label, status, detail in results:
    print(f"{label:<{width}}  {status:<8} {detail}")


### VectorDB Instance

In [ ]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
# chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [ ]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
# embedding_fn = embedding_functions.OpenAIEmbeddingFunction()

In [ ]:
# TODO: Create a collection
# Choose any name you want
# collection = chroma_client.create_collection(
#    name="udaplay",
#    embedding_function=embedding_fn
#)

### Add documents

In [ ]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )